# 05 - Fusion

Validates branch outputs and creates a fusion training matrix by joining on `claim_id`.

In [1]:
import pandas as pd
from functools import reduce

paths = {
    "tabular": "../data/tabular_embeddings.csv",
    "temporal": "../data/temporal_embeddings.csv",
    "graph": "../data/graph_embeddings.csv",
}
branches = {name: pd.read_csv(path) for name, path in paths.items()}

for name, frame in branches.items():
    assert "claim_id" in frame.columns, f"{name} is missing claim_id"
    assert "fraudfound_p" in frame.columns, f"{name} is missing fraudfound_p"
    assert frame["claim_id"].is_unique, f"{name} claim_id is not unique"
    print(name, frame.shape)

tabular (15420, 20)
temporal (15420, 19)
graph (15420, 35)


In [2]:
fusion = reduce(
    lambda left, right: left.merge(right.drop(columns=["fraudfound_p"]), on="claim_id", how="inner"),
    [branches["tabular"], branches["temporal"], branches["graph"]],
)

expected_rows = len(branches["tabular"])
assert len(fusion) == expected_rows, f"Fusion row mismatch: {len(fusion)} != {expected_rows}"
assert fusion.isna().sum().sum() == 0

print("Fusion matrix is ready in memory:", fusion.shape)
fusion.head()

Fusion matrix is ready in memory: (15420, 70)


,claim_id,tab_emb_000,tab_emb_001,tab_emb_002,tab_emb_003,tab_emb_004,tab_emb_005,tab_emb_006,tab_emb_007,tab_emb_008,...,graph_emb_023,graph_emb_024,graph_emb_025,graph_emb_026,graph_emb_027,graph_emb_028,graph_emb_029,graph_emb_030,graph_emb_031,graph_fraud_probability
0,1,-5.661843,-1.624781,-0.178685,-0.871563,-0.109236,0.056308,0.927970,-1.003001,-0.287508,...,0.173657,0.0,0.0,0.108763,0.076876,0.139259,0.042459,0.214958,0.245582,0.482736
1,2,1.626046,-0.092651,0.849183,-0.871563,-0.109236,-0.396656,-0.959237,-0.568405,1.257655,...,0.197840,0.0,0.0,0.104370,0.075722,0.142167,0.063763,0.244063,0.249141,0.470694
2,3,1.626046,-0.625566,-1.206553,1.130326,1.705570,-1.529066,-0.015633,2.101255,0.544503,...,0.203271,0.0,0.0,0.118112,0.075815,0.134387,0.054765,0.238633,0.264387,0.473058
3,4,2.054745,1.439479,0.849183,1.630799,1.478719,-1.585687,-1.292273,-1.189256,0.306786,...,0.204369,0.0,0.0,0.111857,0.089329,0.142971,0.072743,0.234895,0.262925,0.473997
4,5,1.911845,1.506093,-1.720487,-0.371090,-1.016639,-1.076102,-0.071139,-1.251341,0.663361,...,0.189996,0.0,0.0,0.099899,0.079496,0.138779,0.055064,0.233951,0.247448,0.472597
